# OpenBioDesign - ML-Powered Protein Binder Design Platform

Runs on Google Colab free tier with **real ML models**.

**Before starting:** Runtime > Change runtime type > **T4 GPU**

### Models
- **ESM2-650M**: Binding site detection, sequence scoring, mutation analysis
- **ESMFold**: 3D structure prediction (via ColabFold fork)

### Remote Access
Frontend and backend are exposed via public URLs so you can access from any device.

## Cell 1: Install Dependencies (~3 min)

In [ ]:
import os
import time

# Core Python deps
!pip install -q fastapi uvicorn sqlalchemy pydantic pydantic-settings python-multipart httpx numpy scipy
!pip install -q transformers torch

# Node.js 18 for frontend (apt-get gives ancient v12)
!curl -fsSL https://deb.nodesource.com/setup_18.x | bash - > /dev/null 2>&1 && apt-get install -y -qq nodejs > /dev/null 2>&1
!node --version

# localtunnel for remote access (npm package, not pip)
!npm install -g localtunnel

# cloudflared for reliable frontend tunnel (no warning page)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared

# ESMFold deps (from ColabFold - proven to work on Colab)
!pip install -q omegaconf pytorch_lightning biopython ml_collections einops modelcif
!pip install -q git+https://github.com/NVIDIA/dllogger.git
!pip install -q git+https://github.com/sokrypton/openfold.git

# Remove any conflicting esm package BEFORE installing sokrypton fork
!pip uninstall -y esm fair-esm 2>/dev/null; echo 'done'
!pip install -q git+https://github.com/sokrypton/esm.git

print('All dependencies installed!')

## Cell 2: Clone Repository

In [ ]:
import os

!git clone https://github.com/rager1904/OpenBioDesign.git /content/OpenBioDesign 2>/dev/null || echo 'Using existing repo'

os.chdir('/content/OpenBioDesign/platform/backend')
print(f'Working directory: {os.getcwd()}')

## Cell 3: Check GPU & Load ESM2 (~2.5 GB VRAM)

In [ ]:
import torch

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {gpu} ({vram:.1f} GB)')
else:
    print('WARNING: No GPU! Runtime > Change runtime type > GPU')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

from transformers import AutoModelForMaskedLM, AutoTokenizer

ESM2_MODEL = 'facebook/esm2_t33_650M_UR50D'
print(f'Loading {ESM2_MODEL}...')

tokenizer = AutoTokenizer.from_pretrained(ESM2_MODEL)
esm2_model = AutoModelForMaskedLM.from_pretrained(ESM2_MODEL, attn_implementation='eager').to(device).eval()

params = sum(p.numel() for p in esm2_model.parameters()) / 1e6
print(f'ESM2 loaded! ({params:.0f}M params on {device})')

## Cell 4: Load ESMFold (~1 GB VRAM, ~2 min download)

In [ ]:
import os, time

# Download ESMFold weights if not cached
model_name = 'esmfold.model'
if not os.path.isfile(model_name):
    print('Downloading ESMFold weights (~1 GB)...')
    !apt-get install -qq aria2 > /dev/null 2>&1
    !aria2c -q -x 16 https://colabfold.steineggerlab.workers.dev/esm/esmfold.model &
    while not os.path.isfile(model_name):
        time.sleep(5)
    # Wait for aria2 to finish
    while os.path.isfile(f'{model_name}.aria2'):
        time.sleep(5)
    print('Download complete!')
else:
    print('ESMFold weights already cached')

# Load model
import sys
import torch
import esm

# Ensure esm.Alphabet exists for torch.load unpickling
if not hasattr(esm, 'Alphabet'):
    # The sokrypton fork has Alphabet in esm.data
    try:
        from esm.data import Alphabet
        esm.Alphabet = Alphabet
        sys.modules['esm'].Alphabet = Alphabet
        print('Patched esm.Alphabet from esm.data')
    except ImportError:
        raise RuntimeError(
            'Cannot find Alphabet in esm package.\n'
            'Fix: pip uninstall -y esm fair-esm && pip install -q git+https://github.com/sokrypton/esm.git'
        )

print('Loading ESMFold model...')
esmfold_model = torch.load(model_name, weights_only=False)
esmfold_model.eval().cuda().requires_grad_(False)

if torch.cuda.is_available():
    used = torch.cuda.memory_allocated() / 1e9
    print(f'ESMFold loaded! (VRAM used: {used:.1f} GB)')
else:
    print('ESMFold loaded on CPU')

## Cell 5: Start Backend Server

In [ ]:
import sys
import os
import threading
import time

sys.path.insert(0, os.getcwd())

# Patch ESM2 client to use our pre-loaded model
from openbiodesign.infrastructure.esm2_client import ESM2Client

class PatchedESM2(ESM2Client):
    def __init__(self):
        self.model = esm2_model
        self.tokenizer = tokenizer
        self.device = device
        self._model_loaded = True

ESM2Client._instance = PatchedESM2()
print('ESM2 client patched')

# Patch ESMFold client to use our pre-loaded model
from openbiodesign.infrastructure.esmfold_client import ESMFoldClient

class PatchedESMFold(ESMFoldClient):
    def __init__(self):
        self.model = esmfold_model
        self.device = device
        self._model_loaded = True

ESMFoldClient._instance = PatchedESMFold()
print('ESMFold client patched')

# Start FastAPI server in SAME process so singleton patches work
import uvicorn
from openbiodesign.main import app

def run_server():
    uvicorn.run(app, host='0.0.0.0', port=8000, log_level='warning')

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(3)

import httpx
try:
    r = httpx.get('http://localhost:8000/api/v1/health', timeout=10)
    print(f'Backend running! Health: {r.json()}')
except Exception as e:
    print(f'Backend check failed: {e}')

## Cell 6: Expose Backend via Tunnel & Build Frontend

In [ ]:
import subprocess
import time
import os

# --- Create backend tunnel ---
print('Creating backend tunnel...')
backend_tunnel = subprocess.Popen(
    ['npx', 'localtunnel', '--port', '8000'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True
)
time.sleep(5)

backend_url = ''
for _ in range(20):
    line = backend_tunnel.stdout.readline()
    if 'loca.lt' in line or 'https' in line:
        backend_url = line.strip()
        break
    time.sleep(0.5)

if not backend_url:
    backend_url = backend_tunnel.stdout.readline().strip()

api_base = backend_url.rstrip('/') + '/api/v1'
print(f'Backend URL: {backend_url}')
print(f'API base: {api_base}')

# --- Build frontend ---
frontend_dir = '/content/OpenBioDesign/platform/frontend'
os.chdir(frontend_dir)

!npm install

env = os.environ.copy()
env['NEXT_PUBLIC_API_BASE_URL'] = api_base

print('Building frontend...')
result = subprocess.run(['npm', 'run', 'build'], env=env, capture_output=True, text=True, cwd=frontend_dir)
if result.returncode != 0:
    print(f'Build issues: {result.stderr[:500]}')
else:
    print('Frontend built!')

## Cell 7: Start Frontend & Create Tunnel

In [ ]:
import subprocess
import time
import os

env = os.environ.copy()
env['NEXT_PUBLIC_API_BASE_URL'] = api_base

# Start Next.js
frontend_server = subprocess.Popen(
    ['npx', 'next', 'start', '-H', '0.0.0.0', '-p', '3000'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE, env=env,
    cwd='/content/OpenBioDesign/platform/frontend'
)

# Wait for Next.js to be ready
for i in range(20):
    time.sleep(1)
    try:
        httpx.get('http://localhost:3000', timeout=2)
        print(f'Frontend ready after {i+1}s')
        break
    except Exception:
        if i == 19:
            print('Frontend may still be starting...')

# --- Create frontend tunnel via Cloudflare (no warning page) ---
print('Creating frontend tunnel via Cloudflare...')
frontend_tunnel = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:3000'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)

frontend_url = ''
for _ in range(30):
    line = frontend_tunnel.stdout.readline()
    if 'trycloudflare.com' in line:
        frontend_url = line.strip()
        # Extract just the URL
        for part in frontend_url.split():
            if 'trycloudflare.com' in part:
                frontend_url = part
                break
        break
    time.sleep(1)

print()
print('=' * 60)
print('  OPENBIODESIGN - REMOTE ACCESS')
print('=' * 60)
print(f'  Frontend: {frontend_url}')
print(f'  Backend:  {backend_url}')
print()
print('  Open the Frontend URL on any device.')
print('  No warning page - opens directly.')
print('=' * 60)

## Cell 8: Demo - Binding Site Detection (ESM2)

In [ ]:
import httpx

TARGET_SEQUENCE = (
    'MRPSGTAGAALLALLAALCPASRALEEKKVCQGTSNKLTQLGTFEDHFLSLQRMFNNCEVVLGNLEITYVQRNYDLSFLKTIQEVAGYVLIALNTVERIPLENLQIIR'
    'GNMYYENSYALAVLSNINDFNATHTKKEGYGTVIKWVPESGALKKETAAFKKEGYGTVIKWVPESGALKKETAAFK'
)

print(f'Target: EGFR ({len(TARGET_SEQUENCE)} residues)\n')

r = httpx.post(
    'http://localhost:8000/api/v1/esm2/detect-binding-sites',
    json={'sequence': TARGET_SEQUENCE, 'top_k': 8},
    timeout=60.0
)

if r.status_code == 200:
    result = r.json()
    print('=== ESM2 Binding Site Detection ===')
    print(f'Residues: {result["residue_positions_1indexed"]}')
    print(f'Confidence: {result["confidence"]:.4f}')
    print(f'Method: {result["method"]}')
else:
    print(f'Error {r.status_code}: {r.text[:300]}')

In [ ]:
r = httpx.post(
    'http://localhost:8000/api/v1/esm2/score-sequence',
    json={'sequence': TARGET_SEQUENCE},
    timeout=60.0
)

if r.status_code == 200:
    result = r.json()
    print('=== Sequence Fitness Score ===')
    print(f'Mean log-likelihood: {result["mean_log_likelihood"]:.4f}')
    print(f'Interpretation: {result["interpretation"]}')
else:
    print(f'Error {r.status_code}: {r.text[:300]}')

## Cell 9: Demo - Binder Design Workflow

In [ ]:
r = httpx.post(
    'http://localhost:8000/api/v1/workflows/binder-design',
    json={
        'project_id': 'demo-project',
        'target': {
            'name': 'EGFR',
            'sequence': TARGET_SEQUENCE,
            'organism': 'Homo sapiens'
        },
        'hypothesis': 'Design protein binder for EGFR kinase domain',
        'requested_candidates': 3,
        'random_seed': 42
    },
    headers={'Authorization': 'Bearer dev-scientist-key'},
    timeout=120.0
)

if r.status_code == 200:
    result = r.json()
    print('=== Binder Design Results ===')
    print(f'Candidates: {len(result["candidates"])}')
    for i, c in enumerate(result['candidates']):
        print(f'\nCandidate {i+1}:')
        print(f'  Binding: {c["binding_score"]}')
        print(f'  Stability: {c["stability_score"]}')
        print(f'  Sequence: {c["sequence"][:50]}...')
else:
    print(f'Error {r.status_code}: {r.text[:500]}')

## Cell 10: Demo - Structure Prediction (ESMFold)

In [ ]:
if 'result' in dir() and result.get('candidates'):
    seq = result['candidates'][0]['sequence']
    print(f'Predicting structure ({len(seq)} residues)...\n')

    r = httpx.post(
        'http://localhost:8000/api/v1/esmfold/predict',
        json={'sequence': seq},
        timeout=120.0
    )

    if r.status_code == 200:
        s = r.json()
        print('=== ESMFold Structure Prediction ===')
        print(f'Mean pLDDT: {s["mean_plddt"]:.2f}')
        print(f'Confidence: {s["confidence_classification"]}')
        with open('predicted_structure.pdb', 'w') as f:
            f.write(s['pdb_content'])
        print('PDB saved: predicted_structure.pdb')
    else:
        print(f'Error {r.status_code}: {r.text[:300]}')
else:
    print('Run Cell 9 first to generate candidates.')

## Cell 11: Demo - Mutation Analysis

In [ ]:
test_seq = 'ACDEFGHIKLMNPQRSTVWYACDEFGHIKLMNPQRSTVWY'

print('=== Mutation Effect Prediction ===\n')

for pos, wt, mut in [(5, 'A', 'D'), (10, 'K', 'E'), (15, 'P', 'G')]:
    r = httpx.post(
        'http://localhost:8000/api/v1/esm2/predict-mutation',
        json={'sequence': test_seq, 'position': pos, 'mutant_residue': mut},
        timeout=60.0
    )
    if r.status_code == 200:
        m = r.json()
        print(f'{wt}{pos+1}{mut}: delta={m["delta_score"]:.4f} ({m["effect_classification"]})')
    else:
        print(f'{wt}{pos+1}{mut}: Error {r.status_code}')

## Summary

All services running with real ML models:
- **ESM2**: Binding sites, sequence scoring, mutation analysis
- **ESMFold**: Structure prediction with pLDDT confidence
- **Frontend**: Full UI accessible from any device via tunnel URL
- **Backend**: FastAPI with 16+ endpoints

### Resources
- [ESM2 Paper](https://proceedings.mlr.press/v162/esm2.html)
- [ESMFold Paper](https://www.biorxiv.org/content/10.1101/2022.07.20.500901v2)
- [OpenBioDesign](https://github.com/rager1904/OpenBioDesign)